# 06 — Model Inspection

**Goal:** Understand the YOLO26 architecture in detail for future extensibility.

This notebook:
1. Prints the full model structure
2. Identifies backbone, neck, and detect head boundaries
3. Maps layer indices to module types
4. Explains which features are useful for lane marking integration
5. Provides a reference for all layer names and parameter counts

## 1 — Setup

In [ ]:
!pip install -q ultralytics>=8.4.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn as nn

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"

# Load the fine-tuned model (or pretrained for inspection)
WEIGHTS_PATH = os.path.join(ECOCAR_ROOT, "weights", "bdd100k_yolo26s_best.pt")

# Fallback to pretrained if trained weights not available
USE_PRETRAINED = not os.path.isfile(WEIGHTS_PATH)
if USE_PRETRAINED:
    WEIGHTS_PATH = "yolo26s.pt"
    print("⚠️ Trained weights not found. Using COCO-pretrained for inspection.")
    print("   (Architecture is identical; only class count differs.)")
else:
    print(f"✅ Using trained weights: {WEIGHTS_PATH}")

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS_PATH)
inner = model.model  # The nn.Module

## 2 — Model Summary

In [ ]:
total_params = sum(p.numel() for p in inner.parameters())
trainable_params = sum(p.numel() for p in inner.parameters() if p.requires_grad)
num_modules = sum(1 for _ in inner.modules())

print("="*60)
print(" YOLO26 MODEL SUMMARY")
print("="*60)
print(f" Task:              {model.task}")
print(f" Classes:           {len(model.names)}")
print(f" Total parameters:  {total_params:,}")
print(f" Trainable params:  {trainable_params:,}")
print(f" Total modules:     {num_modules}")
print(f" Weights file:      {WEIGHTS_PATH}")
print("="*60)

## 3 — Full Architecture (Top-Level Layers)

In [ ]:
# Get the sequential model layers
try:
    layers = list(inner.model)
except (TypeError, AttributeError):
    layers = list(inner.children())

print(f"\n{'Idx':>4} {'Module Type':<35} {'Params':>12} {'Output Shape (approx)'}")
print("─" * 80)

for idx, layer in enumerate(layers):
    cls_name = layer.__class__.__name__
    params = sum(p.numel() for p in layer.parameters())
    
    # Try to determine output shape from submodules
    shape_info = ""
    if hasattr(layer, 'cv1'):
        if hasattr(layer.cv1, 'conv'):
            out_ch = layer.cv1.conv.out_channels
            shape_info = f"→ {out_ch} channels"
    elif hasattr(layer, 'conv'):
        out_ch = layer.conv.out_channels
        shape_info = f"→ {out_ch} channels"
    elif 'Detect' in cls_name:
        shape_info = "→ detection output"
    elif 'Upsample' in cls_name:
        shape_info = "→ 2× upscale"
    elif 'Concat' in cls_name:
        shape_info = "→ concatenate"
    
    print(f"  {idx:>2}  {cls_name:<35} {params:>12,}  {shape_info}")

print(f"\nTotal top-level layers: {len(layers)}")

## 4 — Identify Backbone / Neck / Head

In [ ]:
# Classify each layer as backbone, neck, or head
detect_keywords = {'Detect', 'Segment', 'Classify', 'OBB', 'Pose', 'WorldDetect', 'E2EDetect'}

backbone_layers = []
neck_layers = []
head_layers = []

found_neck_indicator = False
found_head = False

for idx, layer in enumerate(layers):
    cls_name = layer.__class__.__name__
    params = sum(p.numel() for p in layer.parameters())
    entry = (idx, cls_name, params)
    
    if cls_name in detect_keywords or 'Detect' in cls_name:
        head_layers.append(entry)
        found_head = True
    elif found_neck_indicator and not found_head:
        neck_layers.append(entry)
    elif cls_name in ('Upsample', 'Concat') or 'Concat' in cls_name:
        neck_layers.append(entry)
        found_neck_indicator = True
    elif not found_neck_indicator and not found_head:
        backbone_layers.append(entry)
    elif not found_head:
        neck_layers.append(entry)

# Print results
def print_component(name, entries, color_emoji):
    total_p = sum(e[2] for e in entries)
    print(f"\n{color_emoji} {name} — {len(entries)} layers, {total_p:,} params")
    print(f"  {'Idx':>4} {'Module':<30} {'Params':>12}")
    print(f"  {'─'*50}")
    for idx, cls_name, params in entries:
        print(f"  {idx:>4} {cls_name:<30} {params:>12,}")

print_component("BACKBONE", backbone_layers, "🔵")
print_component("NECK (FPN/PAN)", neck_layers, "🟢")
print_component("DETECTION HEAD", head_layers, "🔴")

## 5 — Detailed Module Tree

In [ ]:
# Print named modules (first 3 levels for readability)
print("\nDetailed module tree (first 3 levels):")
print("="*70)

for name, module in inner.named_modules():
    depth = name.count('.')
    if depth > 3:
        continue
    indent = "  " * depth
    params = sum(p.numel() for p in module.parameters(recurse=False))
    cls_name = module.__class__.__name__
    param_str = f" ({params:,} params)" if params > 0 else ""
    print(f"{indent}├─ {name or 'root'}: {cls_name}{param_str}")

## 6 — Parameter Distribution

In [ ]:
import matplotlib.pyplot as plt

# Parameters per top-level layer
layer_params = []
layer_labels = []
layer_colors = []

backbone_idxs = {e[0] for e in backbone_layers}
neck_idxs = {e[0] for e in neck_layers}
head_idxs = {e[0] for e in head_layers}

for idx, layer in enumerate(layers):
    params = sum(p.numel() for p in layer.parameters())
    layer_params.append(params)
    layer_labels.append(f"[{idx}] {layer.__class__.__name__}")
    
    if idx in backbone_idxs:
        layer_colors.append('#4A90D9')  # blue
    elif idx in neck_idxs:
        layer_colors.append('#50C878')  # green
    else:
        layer_colors.append('#E74C3C')  # red

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(layer_params)), layer_params, color=layer_colors)
ax.set_xlabel('Layer Index')
ax.set_ylabel('Parameters')
ax.set_title('YOLO26 Parameter Distribution by Layer')
ax.set_xticks(range(len(layer_params)))
ax.set_xticklabels([str(i) for i in range(len(layer_params))], fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4A90D9', label='Backbone'),
    Patch(facecolor='#50C878', label='Neck'),
    Patch(facecolor='#E74C3C', label='Head'),
]
ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

## 7 — Detection Head Details

In [ ]:
# Inspect the detection head in detail
if head_layers:
    head_idx = head_layers[0][0]
    head_module = layers[head_idx]
    
    print(f"\nDetection Head (layer [{head_idx}]):")
    print(f"  Type: {head_module.__class__.__name__}")
    print(f"  Params: {sum(p.numel() for p in head_module.parameters()):,}")
    
    # Print submodules
    print(f"\n  Submodules:")
    for name, mod in head_module.named_modules():
        if name and name.count('.') < 2:
            params = sum(p.numel() for p in mod.parameters(recurse=False))
            print(f"    {name}: {mod.__class__.__name__} ({params:,} params)")
    
    # Number of classes
    if hasattr(head_module, 'nc'):
        print(f"\n  Number of classes (nc): {head_module.nc}")
    if hasattr(head_module, 'nl'):
        print(f"  Number of detection layers (nl): {head_module.nl}")
    if hasattr(head_module, 'no'):
        print(f"  Number of outputs per anchor (no): {head_module.no}")

## 8 — Guidance for Future Lane Marking Integration

### Which features to use for lane marking?

| Feature Source | Resolution | Best For |
|---|---|---|
| **Early backbone** (layers 0-3) | High res, low semantic | Edge/texture detection |
| **Mid backbone** (layers 4-6) | Medium | Shape features |
| **Late backbone** (layers 7-9) | Low res, high semantic | Semantic understanding |
| **Neck features** | Multi-scale | Best overall — fused features |

### Recommended approach:
1. **Hook into neck layers** for multi-scale feature maps
2. **Feed into a lightweight lane decoder** (e.g., simple conv + upsampling head)
3. **Freeze backbone + neck** initially, train only the lane decoder
4. **Fine-tune end-to-end** once the lane head converges

### Key layer indices for lane features:
Use the component identification above to find the exact indices for your model variant.

In [ ]:
# Print a quick reference of recommended hook points
print("\n" + "="*60)
print(" RECOMMENDED FEATURE EXTRACTION POINTS")
print("="*60)

if backbone_layers:
    mid = len(backbone_layers) // 2
    print(f"\n  Backbone (for feature extractor):")
    print(f"    Early: layer [{backbone_layers[0][0]}] ({backbone_layers[0][1]})")
    print(f"    Mid:   layer [{backbone_layers[mid][0]}] ({backbone_layers[mid][1]})")
    print(f"    Late:  layer [{backbone_layers[-1][0]}] ({backbone_layers[-1][1]})")

if neck_layers:
    print(f"\n  Neck (for multi-scale features):")
    for idx, cls_name, params in neck_layers:
        if params > 0:  # Skip Upsample/Concat with no params
            print(f"    layer [{idx}] ({cls_name}, {params:,} params)")

print(f"\n  Detection head:")
for idx, cls_name, params in head_layers:
    print(f"    layer [{idx}] ({cls_name}, {params:,} params)")

print("\n" + "="*60)
print("\n📌 Use notebook 05 to extract features from these layers.")
print("   The FeatureExtractor class accepts layer_indices directly.")

## 9 — Export Model Info to File

In [ ]:
# Save architecture info for reference
info_path = os.path.join(ECOCAR_ROOT, "outputs", "model_architecture_info.txt")
os.makedirs(os.path.dirname(info_path), exist_ok=True)

with open(info_path, 'w') as f:
    f.write("YOLO26 Architecture Reference\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Weights: {WEIGHTS_PATH}\n")
    f.write(f"Task: {model.task}\n")
    f.write(f"Classes: {len(model.names)}\n")
    f.write(f"Total params: {total_params:,}\n")
    f.write(f"Total layers: {len(layers)}\n\n")
    
    f.write("BACKBONE\n")
    for idx, cls_name, params in backbone_layers:
        f.write(f"  [{idx}] {cls_name} ({params:,})\n")
    
    f.write("\nNECK\n")
    for idx, cls_name, params in neck_layers:
        f.write(f"  [{idx}] {cls_name} ({params:,})\n")
    
    f.write("\nHEAD\n")
    for idx, cls_name, params in head_layers:
        f.write(f"  [{idx}] {cls_name} ({params:,})\n")

print(f"✅ Architecture info saved: {info_path}")
print("\n🎯 Model inspection complete!")